# UR3e Gazebo Simulator Control

This notebook talks to the Gazebo simulator running inside the `iscoin_simulator` Docker container.

**Prerequisites:** The simulator must be running:
```bash
cd ~/Documents/HES_Duckify/ur3e-simulator/.docker
xhost +local:docker
docker compose run --rm cpu
# inside container:
ros2 launch iscoin_simulation_gz iscoin_sim_control.launch.py
```

## 1. Check that the simulator is reachable

In [1]:
import subprocess
import json
import math

CONTAINER = "iscoin_simulator"

def docker_exec(cmd, timeout=10):
    """Run a command inside the simulator container and return stdout."""
    result = subprocess.run(
        ["docker", "exec", CONTAINER, "bash", "-c", cmd],
        capture_output=True, text=True, timeout=timeout
    )
    if result.returncode != 0:
        print(f"STDERR: {result.stderr}")
    return result.stdout

# Quick check: list ROS2 topics
print(docker_exec("source /opt/ros/humble/setup.bash && ros2 topic list"))

/clock
/drawing
/dynamic_joint_states
/joint_state_broadcaster/transition_event
/joint_states
/joint_trajectory_controller/controller_state
/joint_trajectory_controller/joint_trajectory
/joint_trajectory_controller/state
/joint_trajectory_controller/transition_event
/parameter_events
/robot_description
/rosout
/tf
/tf_static



## 2. Read current joint positions

In [ ]:
JOINT_NAMES = [
    "shoulder_pan_joint",
    "shoulder_lift_joint",
    "elbow_joint",
    "wrist_1_joint",
    "wrist_2_joint",
    "wrist_3_joint",
]

def read_joint_positions():
    """Read the current 6 joint angles from the simulator."""
    raw = docker_exec(
        "source /opt/ros/humble/setup.bash && "
        "ros2 topic echo /joint_states --once --no-arr",
        timeout=15
    )
    # Parse the YAML-like output
    names = []
    positions = []
    in_names = False
    in_positions = False
    for line in raw.splitlines():
        line = line.strip()
        if line == "name:":
            in_names = True; in_positions = False; continue
        if line == "position:":
            in_positions = True; in_names = False; continue
        if line.startswith("- ") and in_names:
            names.append(line[2:])
        elif line.startswith("- ") and in_positions:
            positions.append(float(line[2:]))
        elif not line.startswith("- "):
            in_names = False; in_positions = False

    all_joints = dict(zip(names, positions))
    result = {}
    for name in JOINT_NAMES:
        if name in all_joints:
            result[name] = all_joints[name]
    return result

joints = read_joint_positions()
print("Current joint positions:")
for name, rad in joints.items():
    print(f"  {name:25s}  {rad:+.4f} rad  ({math.degrees(rad):+.1f}°)")

## 3. Move the robot to a target position

We write a trajectory JSON file, copy it into the container, and use the built-in demo node to execute it.

In [ ]:
import tempfile
import os

def move_to(positions, duration_sec=4):
    """
    Move the robot to target joint positions (list of 6 floats, in radians).
    duration_sec: how long the movement should take.
    """
    traj = {
        "traj0": [
            {
                "positions": positions,
                "velocities": [0.0] * 6,
                "time_from_start": [duration_sec, 0]
            }
        ]
    }

    # Write JSON to a temp file on host, then copy into container
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
        json.dump(traj, f)
        tmp_path = f.name

    subprocess.run(["docker", "cp", tmp_path, f"{CONTAINER}:/tmp/traj.json"], check=True)
    os.unlink(tmp_path)

    # Execute using the demo node
    output = docker_exec(
        "source /root/ws_moveit/install/setup.bash && "
        "ros2 run iscoin_driver demo.py --ros-args -p traj:=/tmp/traj.json",
        timeout=30
    )
    print(output)
    return output

In [ ]:
# Example: move shoulder_pan_joint from ~1.19 to ~1.76 (about 30° rotation)
# These are the same positions from the demo traj0.json
move_to([1.7649, -1.1278, 1.0483, -1.5995, -1.4845, 1.0469], duration_sec=4)

In [ ]:
# Read positions again to confirm the robot moved
joints = read_joint_positions()
print("Joint positions after move:")
for name, rad in joints.items():
    print(f"  {name:25s}  {rad:+.4f} rad  ({math.degrees(rad):+.1f}°)")

In [ ]:
# Move back to the starting position
move_to([1.1945, -1.1268, 1.0484, -1.5988, -1.5214, 1.0469], duration_sec=4)